# 🚀 3DGS Sparse-view Room Reconstruction Worker (FastAPI + DUSt3R/InstantSplat + Cloudflare Tunnel)

💡 **Ưu điểm Cloudflare:** Hoàn toàn **miễn phí**, **không cần đăng ký tài khoản**, **không cần token**, băng thông không giới hạn!

**Lưu ý quan trọng:** Hãy đảm bảo bạn đã bật **T4 GPU**:
> `Runtime` -> `Change runtime type` -> Chọn `T4 GPU` -> `Save`

In [ ]:
# 1. Kiểm tra GPU
!nvidia-smi

# 2. Cài đặt các thư viện Web service cần thiết
!pip install -q fastapi uvicorn python-multipart nest-asyncio aiofiles requests

# 3. Tải và cài đặt Cloudflared (Quick Tunnel miễn phí, không cần đăng nhập)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
# 4. Clone & cài đặt InstantSplat / DUSt3R (Sparse-view 3DGS)
!git clone --recursive https://github.com/NVlabs/InstantSplat.git
%cd InstantSplat

# Cài đặt submodules vi phân Gaussian Rasterization & Simple-KNN
!pip install -q submodules/diff-gaussian-rasterization
!pip install -q submodules/simple-knn

# Cài đặt thư viện DUSt3R
%cd dust3r
!pip install -q -r requirements.txt
!mkdir -p checkpoints
!wget -c https://download.europe.naverlabs.com/dust3r/DUSt3R_ViTlarge_BaseDecoder_512_dpt.pth -P checkpoints/
%cd ..

In [ ]:
# 5. Dựng FastAPI Service & Public qua Cloudflare Tunnel
import os
import shutil
import zipfile
import subprocess
import uuid
import time
import re
import threading
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
import nest_asyncio

nest_asyncio.apply()
app = FastAPI(title="3DGS Reconstruction Worker")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

WORK_DIR = "/content/workspace"
os.makedirs(WORK_DIR, exist_ok=True)

@app.get("/")
def health():
    return {"status": "online", "tunnel": "Cloudflare", "gpu": "T4 GPU Ready"}

@app.post("/reconstruct")
async def reconstruct_scene(file: UploadFile = File(...)):
    if not file.filename.endswith(".zip"):
        raise HTTPException(status_code=400, detail="Vui lòng gửi file nén .zip chứa ảnh")
    
    job_id = str(uuid.uuid4())[:8]
    job_folder = os.path.join(WORK_DIR, job_id)
    input_images_dir = os.path.join(job_folder, "images")
    output_dir = os.path.join(job_folder, "output")
    
    os.makedirs(input_images_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)
    
    zip_path = os.path.join(job_folder, "input.zip")
    with open(zip_path, "wb") as buffer:
        shutil.copyfileobj(file.file, buffer)
        
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(input_images_dir)
        
    # Thu thập danh sách ảnh hợp lệ
    valid_images = []
    for root, _, files in os.walk(input_images_dir):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                valid_images.append(os.path.join(root, f))
                
    if len(valid_images) < 3:
        raise HTTPException(status_code=400, detail=f"Cần ít nhất 3 ảnh (nhận được {len(valid_images)})")
        
    print(f"[{job_id}] Đang xử lý {len(valid_images)} ảnh bằng InstantSplat...")
    
    # Chạy lệnh InstantSplat
    cmd = [
        "python", "run.py",
        "--image_dir", input_images_dir,
        "--output_dir", output_dir,
        "--iterations", "7000",
        "--save_iterations", "7000"
    ]
    
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        print("Lỗi:", proc.stderr)
        raise HTTPException(status_code=500, detail=f"Lỗi pipeline: {proc.stderr[-400:]}")
        
    ply_path = None
    for root, _, files in os.walk(output_dir):
        for f in files:
            if f.endswith(".ply"):
                ply_path = os.path.join(root, f)
                break
                
    if not ply_path:
        raise HTTPException(status_code=500, detail="Không tìm thấy file .ply kết quả")
        
    print(f"[{job_id}] Hoàn thành tái tạo 3DGS! Đang gửi file: {ply_path}")
    return FileResponse(ply_path, filename=f"room_{job_id}.ply", media_type="application/octet-stream")

# Khởi động Cloudflare Tunnel ở background
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

cloudflare_url = None
print("Đang thiết lập Cloudflare Tunnel...")
for _ in range(60):
    line = cf_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        cloudflare_url = match.group(0)
        break
    time.sleep(0.3)

print("\n" + "="*60)
if cloudflare_url:
    print(f"🎉 CLOUDFLARE PUBLIC URL CỦA BẠN: {cloudflare_url}")
else:
    print("Chưa lấy được link tự động, hãy kiểm tra log cloudflared phía dưới.")
print("="*60 + "\n")

# Chạy uvicorn server
uvicorn.run(app, host="0.0.0.0", port=8000)